# rnn_walk_forward_validation_tuning

Recurrent neural network validation tuning using the full-history session-aligned dataset.

This notebook uses compact GRU/LSTM sequence models. Instead of giving the model a single engineered row, it builds rolling ticker-specific sequences ending on the prediction date. The default grid is intentionally smaller than the SVM/RF/logistic grids because each candidate requires neural-network training.

The pipeline keeps the same evaluation discipline as the other notebooks: it splits on the full session calendar before removing neutral targets, selects feature/hyperparameter configurations with expanding-window walk-forward validation, calibrates the final probability threshold on the holdout validation period, and evaluates the frozen model on the untouched test split.


In [ ]:
from __future__ import annotations

import os
from pathlib import Path

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score

try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers, regularizers
except ImportError as exc:
    tf = None
    keras = None
    layers = None
    regularizers = None
    TENSORFLOW_IMPORT_ERROR = exc
else:
    TENSORFLOW_IMPORT_ERROR = None

if tf is None:
    raise ImportError(
        "TensorFlow is required for this RNN notebook. Update the project environment from environment.yml "
        "or install into the active notebook kernel with: python -m pip install \"tensorflow>=2.16,<2.18\""
    ) from TENSORFLOW_IMPORT_ERROR

pd.set_option("display.max_columns", 220)
pd.set_option("display.width", 280)


In [ ]:
from __future__ import annotations

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data" / "datasets").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from notebook_utils.feature_set_grid_builder import FeatureFrameBuilder, FeatureSetGridBuilder
from notebook_utils.metrics import ClassificationMetrics
from notebook_utils.model_report_builder import ModelReportBuilder
from notebook_utils.split_utils import make_split_dates, make_walk_forward_fold_specs, subset_by_dates

CONFIG = {
    "dataset_path": PROJECT_ROOT / "data" / "datasets" / "stock_panel_nine_tickers_session_aligned_full_history_adjusted_google_score_raw.csv",
    "excluded_tickers": ["NFLX"],
    "neutral_band": 0.005,
    "test_size": 0.25,
    "validation_fraction_within_pretest": 0.25,
    "min_validation_dates": 20,
    "gap_days": 1,
    "random_state": 42,
    "selection_metric": "selection_score",
    "primary_validation_metric": "balanced_accuracy",
    "walk_forward_folds": 4,
    "walk_forward_validation_dates": 80,
    "walk_forward_min_train_dates": 252,
    "walk_forward_stability_penalty": 0.25,
    "max_features_per_model": 10,
    "tune_decision_threshold": True,
    "threshold_min_quantile": 0.05,
    "threshold_max_quantile": 0.95,
    "threshold_grid_size": 181,
    "bootstrap_iterations": 1000,
    "bootstrap_ci": 0.95,
    "early_stopping_fraction": 0.15,
    "min_early_stopping_dates": 40,
    "min_fit_dates": 120,
}

RNN_PARAM_GRID = [
    {
        "param_set": "gru_len5_h8_drop0p2_l2_1e-3_lr1e-3_b128_balanced",
        "sequence_length": 5,
        "cell_type": "GRU",
        "hidden_units": 8,
        "dropout": 0.2,
        "recurrent_dropout": 0.0,
        "dense_units": 0,
        "dense_dropout": 0.0,
        "l2": 1e-3,
        "learning_rate": 1e-3,
        "batch_size": 128,
        "max_epochs": 80,
        "patience": 8,
        "class_weight_mode": "balanced",
    },
    {
        "param_set": "gru_len10_h8_drop0p2_l2_1e-3_lr1e-3_b128_balanced",
        "sequence_length": 10,
        "cell_type": "GRU",
        "hidden_units": 8,
        "dropout": 0.2,
        "recurrent_dropout": 0.0,
        "dense_units": 0,
        "dense_dropout": 0.0,
        "l2": 1e-3,
        "learning_rate": 1e-3,
        "batch_size": 128,
        "max_epochs": 80,
        "patience": 8,
        "class_weight_mode": "balanced",
    },
    {
        "param_set": "gru_len20_h16_drop0p3_l2_1e-3_lr5e-4_b128_balanced",
        "sequence_length": 20,
        "cell_type": "GRU",
        "hidden_units": 16,
        "dropout": 0.3,
        "recurrent_dropout": 0.0,
        "dense_units": 0,
        "dense_dropout": 0.0,
        "l2": 1e-3,
        "learning_rate": 5e-4,
        "batch_size": 128,
        "max_epochs": 100,
        "patience": 10,
        "class_weight_mode": "balanced",
    },
    {
        "param_set": "lstm_len10_h8_drop0p2_l2_1e-3_lr1e-3_b128_balanced",
        "sequence_length": 10,
        "cell_type": "LSTM",
        "hidden_units": 8,
        "dropout": 0.2,
        "recurrent_dropout": 0.0,
        "dense_units": 0,
        "dense_dropout": 0.0,
        "l2": 1e-3,
        "learning_rate": 1e-3,
        "batch_size": 128,
        "max_epochs": 80,
        "patience": 8,
        "class_weight_mode": "balanced",
    },
    {
        "param_set": "gru_len10_h16_dense8_drop0p3_l2_1e-2_lr5e-4_b128_balanced",
        "sequence_length": 10,
        "cell_type": "GRU",
        "hidden_units": 16,
        "dropout": 0.3,
        "recurrent_dropout": 0.0,
        "dense_units": 8,
        "dense_dropout": 0.2,
        "l2": 1e-2,
        "learning_rate": 5e-4,
        "batch_size": 128,
        "max_epochs": 100,
        "patience": 10,
        "class_weight_mode": "balanced",
    },
    {
        "param_set": "lstm_len20_h16_dense8_drop0p3_l2_1e-2_lr5e-4_b128_balanced",
        "sequence_length": 20,
        "cell_type": "LSTM",
        "hidden_units": 16,
        "dropout": 0.3,
        "recurrent_dropout": 0.0,
        "dense_units": 8,
        "dense_dropout": 0.2,
        "l2": 1e-2,
        "learning_rate": 5e-4,
        "batch_size": 128,
        "max_epochs": 100,
        "patience": 10,
        "class_weight_mode": "balanced",
    },
]

pd.DataFrame(RNN_PARAM_GRID)

feature_grid = FeatureSetGridBuilder.build(
    max_features_per_model=CONFIG["max_features_per_model"],
)

PRICE_FEATURES = feature_grid.price_features
VOLUME_FEATURE_OPTIONS = feature_grid.volume_feature_options
GDELT_FEATURE_OPTIONS = feature_grid.gdelt_feature_options
GDELT_SENTIMENT_FEATURE_OPTIONS = feature_grid.gdelt_sentiment_feature_options
GDELT_ATTENTION_FEATURE_OPTIONS = feature_grid.gdelt_attention_feature_options
REDDIT_FEATURE_OPTIONS = feature_grid.reddit_feature_options
REDDIT_ATTENTION_FEATURE_OPTIONS = feature_grid.reddit_attention_feature_options
GOOGLE_TRENDS_FEATURE_OPTIONS = feature_grid.google_trends_feature_options
GOOGLE_SCORE_ATTENTION_FEATURE_OPTIONS = feature_grid.google_score_attention_feature_options
DERIVED_FEATURE_COLUMNS = feature_grid.derived_feature_columns
BASE_VOLUME_OPTION = feature_grid.base_volume_option
BASELINE_FEATURE_SET = feature_grid.baseline_feature_set
FEATURE_SET_SPECS = feature_grid.feature_set_specs
FEATURE_SETS = feature_grid.feature_sets
FEATURE_SET_METADATA = feature_grid.feature_set_metadata
SKIPPED_FEATURE_SETS = feature_grid.skipped_feature_sets
FEATURE_SETS_TO_TEST = feature_grid.feature_sets_to_test
pd.DataFrame(RNN_PARAM_GRID)


In [ ]:
from __future__ import annotations


def safe_auc(y_true: pd.Series, scores: np.ndarray) -> float:
    if pd.Series(y_true).nunique() < 2:
        return np.nan
    return float(roc_auc_score(y_true, scores))


In [ ]:
raw_df = pd.read_csv(CONFIG["dataset_path"], parse_dates=["date"])
raw_df = raw_df[~raw_df["ticker"].isin(CONFIG["excluded_tickers"])].copy()
raw_df = raw_df.sort_values(["ticker", "date"]).reset_index(drop=True)

feature_df = FeatureFrameBuilder.build_feature_frame(raw_df, neutral_band=CONFIG["neutral_band"])
train_dates, validation_dates, test_dates = make_split_dates(
    feature_df,
    test_size=CONFIG["test_size"],
    validation_fraction_within_pretest=CONFIG["validation_fraction_within_pretest"],
    min_validation_dates=CONFIG["min_validation_dates"],
    gap_days=CONFIG["gap_days"],
)
walk_forward_fold_specs = make_walk_forward_fold_specs(
    train_dates,
    n_folds=CONFIG["walk_forward_folds"],
    validation_size=CONFIG["walk_forward_validation_dates"],
    min_train_dates=CONFIG["walk_forward_min_train_dates"],
    gap_days=CONFIG["gap_days"],
)

split_date_map = {
    "train": train_dates,
    "validation": validation_dates,
    "test": test_dates,
}
feature_df["split"] = "gap"
for split_name, split_dates in split_date_map.items():
    feature_df.loc[feature_df["date"].isin(split_dates), "split"] = split_name

modeled_df = feature_df[feature_df["target"].isin([0.0, 1.0])].copy()
modeled_df["target"] = modeled_df["target"].astype(int)

train_df = subset_by_dates(modeled_df, train_dates)
validation_df = subset_by_dates(modeled_df, validation_dates)
test_df = subset_by_dates(modeled_df, test_dates)

split_summary_rows = []
for split_name in ["train", "validation", "test"]:
    all_split_df = feature_df[feature_df["split"].eq(split_name)]
    modeled_split_df = modeled_df[modeled_df["split"].eq(split_name)]
    split_summary_rows.append(
        {
            "split": split_name,
            "session_rows": len(all_split_df),
            "modeled_rows": len(modeled_split_df),
            "session_dates": all_split_df["date"].nunique(),
            "modeled_dates": modeled_split_df["date"].nunique(),
            "date_min": all_split_df["date"].min(),
            "date_max": all_split_df["date"].max(),
            "target_positive_rate": modeled_split_df["target"].mean(),
        }
    )

split_summary_df = pd.DataFrame(split_summary_rows)
walk_forward_fold_summary_df = pd.DataFrame(
    [
        {
            "fold": spec["fold"],
            "train_n_dates": spec["train_n_dates"],
            "train_date_min": spec["train_date_min"],
            "train_date_max": spec["train_date_max"],
            "validation_n_dates": spec["validation_n_dates"],
            "validation_date_min": spec["validation_date_min"],
            "validation_date_max": spec["validation_date_max"],
        }
        for spec in walk_forward_fold_specs
    ]
)

split_summary_df


In [ ]:
walk_forward_fold_summary_df

In [ ]:
neutral_summary_by_ticker_df = (
    feature_df.groupby("ticker")
    .agg(
        rows=("target_available", "size"),
        target_available=("target_available", "sum"),
        neutral=("is_neutral", "sum"),
    )
    .reset_index()
)
neutral_summary_by_ticker_df["neutral_rate_among_available"] = (
    neutral_summary_by_ticker_df["neutral"] / neutral_summary_by_ticker_df["target_available"]
)
neutral_summary_by_ticker_df["modeled_rate_among_available"] = 1.0 - neutral_summary_by_ticker_df["neutral_rate_among_available"]

neutral_summary_by_split_df = (
    feature_df[feature_df["split"].isin(["train", "validation", "test"])]
    .groupby("split")
    .agg(
        rows=("target_available", "size"),
        target_available=("target_available", "sum"),
        neutral=("is_neutral", "sum"),
    )
    .reindex(["train", "validation", "test"])
    .reset_index()
)
neutral_summary_by_split_df["neutral_rate_among_available"] = (
    neutral_summary_by_split_df["neutral"] / neutral_summary_by_split_df["target_available"]
)
neutral_summary_by_split_df["modeled_rate_among_available"] = 1.0 - neutral_summary_by_split_df["neutral_rate_among_available"]

print("Neutral coverage by split")
print(neutral_summary_by_split_df.to_string(index=False))
print("\nNeutral coverage by ticker")
neutral_summary_by_ticker_df

In [ ]:
missing_requested_feature_sets = [name for name in RNN_FEATURE_SETS_TO_TEST if name not in FEATURE_SETS]
if missing_requested_feature_sets:
    raise KeyError(f"Unknown feature sets: {missing_requested_feature_sets}")

missing_feature_columns = sorted(
    {
        feature
        for name in FEATURE_SETS_TO_TEST
        for feature in FEATURE_SETS[name]
        if feature not in feature_df.columns and feature not in DERIVED_FEATURE_COLUMNS
    }
)
if missing_feature_columns:
    raise KeyError(f"Missing feature columns: {missing_feature_columns}")

candidate_feature_sets_df = pd.DataFrame(
    [
        {
            "feature_set": feature_set_name,
            "feature_family": FEATURE_SET_METADATA[feature_set_name]["feature_family"],
            "n_features": len(FEATURE_SETS[feature_set_name]),
            "features": FEATURE_SETS[feature_set_name],
        }
        for feature_set_name in FEATURE_SETS_TO_TEST
    ]
).sort_values(["feature_family", "n_features", "feature_set"]).reset_index(drop=True)

all_available_feature_sets_df = pd.DataFrame(
    [
        {
            "feature_set": feature_set_name,
            "feature_family": FEATURE_SET_METADATA[feature_set_name]["feature_family"],
            "n_features": len(features),
            "features": features,
        }
        for feature_set_name, features in FEATURE_SETS.items()
    ]
).sort_values(["feature_family", "n_features", "feature_set"]).reset_index(drop=True)

skipped_feature_sets_df = pd.DataFrame(SKIPPED_FEATURE_SETS)

print(f"Selection metric: {CONFIG['selection_metric']}")
print(f"Primary validation metric: {CONFIG['primary_validation_metric']}")
print(f"Walk-forward folds: {len(walk_forward_fold_specs)}")
print(f"Walk-forward stability penalty: {CONFIG['walk_forward_stability_penalty']}")
print(f"Tune decision threshold in each validation fold: {CONFIG['tune_decision_threshold']}")
print(f"Max features per model: {CONFIG['max_features_per_model']}")
print(f"Available feature sets: {len(FEATURE_SETS)}")
print(f"RNN feature sets to test: {len(FEATURE_SETS_TO_TEST)}")
print(f"Attention feature sets to test: {sum('attention' in FEATURE_SET_METADATA[name]['feature_family'] for name in FEATURE_SETS_TO_TEST)}")
print(f"Skipped feature sets above max feature limit: {len(SKIPPED_FEATURE_SETS)}")
print(f"RNN parameter sets: {len(RNN_PARAM_GRID)}")
print(f"Walk-forward validation fits: {len(FEATURE_SETS_TO_TEST) * len(RNN_PARAM_GRID) * len(walk_forward_fold_specs)}")

candidate_feature_sets_df


In [ ]:
from __future__ import annotations


RNN_PARAM_COLUMNS = [
    "sequence_length",
    "cell_type",
    "hidden_units",
    "dropout",
    "recurrent_dropout",
    "dense_units",
    "dense_dropout",
    "l2",
    "learning_rate",
    "batch_size",
    "max_epochs",
    "patience",
    "class_weight_mode",
]


class SequencePreprocessor:
    def fit(self, X: np.ndarray) -> "SequencePreprocessor":
        flat = X.reshape(-1, X.shape[-1]).astype(float)
        self.median_ = np.nanmedian(flat, axis=0)
        self.median_ = np.where(np.isfinite(self.median_), self.median_, 0.0)
        filled = np.where(np.isnan(flat), self.median_, flat)
        self.mean_ = filled.mean(axis=0)
        self.std_ = filled.std(axis=0)
        self.std_ = np.where((self.std_ > 0.0) & np.isfinite(self.std_), self.std_, 1.0)
        return self

    def transform(self, X: np.ndarray) -> np.ndarray:
        filled = np.where(np.isnan(X), self.median_, X)
        scaled = (filled - self.mean_) / self.std_
        return scaled.astype("float32")

    def fit_transform(self, X: np.ndarray) -> np.ndarray:
        return self.fit(X).transform(X)


def prepare_train_eval_feature_frames(
    features: list[str],
    train_input_df: pd.DataFrame,
    eval_input_df: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    if "google_trends_above_ticker_train_median" in features:
        return FeatureFrameBuilder.add_google_trends_train_median_feature(train_input_df, eval_input_df)
    return train_input_df, eval_input_df


def params_from_result_row(row: dict | pd.Series) -> dict:
    return {
        "param_set": row["param_set"],
        "sequence_length": int(row["sequence_length"]),
        "cell_type": row["cell_type"],
        "hidden_units": int(row["hidden_units"]),
        "dropout": float(row["dropout"]),
        "recurrent_dropout": float(row["recurrent_dropout"]),
        "dense_units": int(row["dense_units"]),
        "dense_dropout": float(row["dense_dropout"]),
        "l2": float(row["l2"]),
        "learning_rate": float(row["learning_rate"]),
        "batch_size": int(row["batch_size"]),
        "max_epochs": int(row["max_epochs"]),
        "patience": int(row["patience"]),
        "class_weight_mode": row.get("class_weight_mode", "balanced"),
    }


def class_weight_from_target(y_train: np.ndarray, mode: str | None) -> dict[int, float] | None:
    if mode in [None, "none"]:
        return None
    counts = pd.Series(y_train).value_counts()
    n_total = float(len(y_train))
    n_negative = float(counts.get(0, 0.0))
    n_positive = float(counts.get(1, 0.0))
    if n_negative <= 0.0 or n_positive <= 0.0:
        return None
    return {0: n_total / (2.0 * n_negative), 1: n_total / (2.0 * n_positive)}


def split_train_for_early_stopping(train_input_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    dates = np.array(sorted(train_input_df["date"].unique()))
    n_dates = len(dates)
    n_validation = max(
        CONFIG["min_early_stopping_dates"],
        int(round(n_dates * CONFIG["early_stopping_fraction"])),
    )
    n_validation = min(n_validation, max(1, n_dates - CONFIG["min_fit_dates"]))
    fit_dates = dates[:-n_validation]
    early_stop_dates = dates[-n_validation:]
    return subset_by_dates(train_input_df, fit_dates), subset_by_dates(train_input_df, early_stop_dates)


def make_sequence_dataset(
    sequence_source_df: pd.DataFrame,
    sample_df: pd.DataFrame,
    features: list[str],
    sequence_length: int,
) -> tuple[np.ndarray, np.ndarray, pd.DataFrame]:
    source_lookup = {}
    for ticker, ticker_df in sequence_source_df.sort_values(["ticker", "date"]).groupby("ticker", sort=False):
        dates = [pd.Timestamp(value) for value in ticker_df["date"]]
        source_lookup[ticker] = {
            "positions": {date: position for position, date in enumerate(dates)},
            "values": ticker_df[features].to_numpy(dtype=float),
        }

    sequences = []
    targets = []
    metadata_rows = []
    for row in sample_df.sort_values(["date", "ticker"]).itertuples(index=False):
        ticker_data = source_lookup.get(row.ticker)
        if ticker_data is None:
            continue
        position = ticker_data["positions"].get(pd.Timestamp(row.date))
        if position is None:
            continue
        start = position - sequence_length + 1
        if start < 0:
            continue
        sequences.append(ticker_data["values"][start : position + 1])
        targets.append(int(row.target))
        metadata_rows.append({"date": row.date, "ticker": row.ticker, "target": int(row.target)})

    if not sequences:
        raise ValueError(f"No sequences were built for sequence_length={sequence_length}")
    return np.stack(sequences).astype(float), np.asarray(targets, dtype=int), pd.DataFrame(metadata_rows)


def build_rnn_model(params: dict, n_features: int) -> keras.Model:
    if tf is None:
        raise ImportError(
            "TensorFlow is not installed in this Python environment. Install TensorFlow before running the RNN notebook."
        ) from TENSORFLOW_IMPORT_ERROR

    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(CONFIG["random_state"])
    regularizer = regularizers.l2(params["l2"]) if params["l2"] > 0 else None
    inputs = keras.Input(shape=(params["sequence_length"], n_features))
    cell_kwargs = {
        "units": params["hidden_units"],
        "dropout": params["dropout"],
        "recurrent_dropout": params["recurrent_dropout"],
        "kernel_regularizer": regularizer,
        "recurrent_regularizer": regularizer,
    }
    if params["cell_type"] == "GRU":
        x = layers.GRU(**cell_kwargs)(inputs)
    elif params["cell_type"] == "LSTM":
        x = layers.LSTM(**cell_kwargs)(inputs)
    else:
        raise ValueError(f"Unsupported recurrent cell type: {params['cell_type']}")

    if params["dense_units"] > 0:
        x = layers.Dense(params["dense_units"], activation="relu", kernel_regularizer=regularizer)(x)
        if params["dense_dropout"] > 0:
            x = layers.Dropout(params["dense_dropout"])(x)

    outputs = layers.Dense(1, activation="sigmoid")(x)
    model = keras.Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=params["learning_rate"]),
        loss="binary_crossentropy",
    )
    return model


def candidate_thresholds_from_scores(scores: np.ndarray) -> np.ndarray:
    finite_scores = np.asarray(scores, dtype=float)
    finite_scores = finite_scores[np.isfinite(finite_scores)]
    if len(finite_scores) == 0:
        return np.array([0.5])
    quantiles = np.linspace(
        CONFIG["threshold_min_quantile"],
        CONFIG["threshold_max_quantile"],
        CONFIG["threshold_grid_size"],
    )
    thresholds = np.quantile(finite_scores, quantiles)
    return np.unique(np.r_[thresholds, 0.5])


def best_threshold_for_balanced_accuracy(y_true: pd.Series | np.ndarray, scores: np.ndarray) -> tuple[float, float]:
    rows = []
    for threshold in candidate_thresholds_from_scores(scores):
        preds = (scores >= threshold).astype(int)
        rows.append((float(threshold), float(balanced_accuracy_score(y_true, preds))))
    return max(rows, key=lambda item: item[1])


def metrics_from_scores(y_true: pd.Series | np.ndarray, scores: np.ndarray, threshold: float) -> dict:
    metrics = ClassificationMetrics.metrics_from_scores(y_true, scores, threshold)
    preds = metrics.pop("preds")
    return {
        "preds": preds,
        "predicted_positive_rate": float(np.mean(preds)),
        **metrics,
    }



def add_rnn_param_columns(row: dict, params: dict) -> None:
    for column in RNN_PARAM_COLUMNS:
        row[column] = params.get(column)


def evaluate_rnn_params(
    *,
    feature_set_name: str,
    features: list[str],
    params: dict,
    train_input_df: pd.DataFrame,
    eval_input_df: pd.DataFrame,
    split_name: str,
    decision_threshold: float | None = None,
    tune_threshold: bool = False,
    return_predictions: bool = False,
) -> dict | tuple[dict, pd.DataFrame]:
    train_features_df, eval_features_df = prepare_train_eval_feature_frames(
        features,
        train_input_df,
        eval_input_df,
    )
    fit_input_df, early_stop_input_df = split_train_for_early_stopping(train_features_df)

    sequence_length = params["sequence_length"]
    X_fit_raw, y_fit, _ = make_sequence_dataset(feature_df, fit_input_df, features, sequence_length)
    X_early_raw, y_early, _ = make_sequence_dataset(feature_df, early_stop_input_df, features, sequence_length)
    X_eval_raw, y_eval, eval_metadata_df = make_sequence_dataset(feature_df, eval_features_df, features, sequence_length)

    preprocessor = SequencePreprocessor()
    X_fit = preprocessor.fit_transform(X_fit_raw)
    X_early = preprocessor.transform(X_early_raw)
    X_eval = preprocessor.transform(X_eval_raw)

    model = build_rnn_model(params, n_features=len(features))
    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=params["patience"],
            restore_best_weights=True,
        )
    ]
    history = model.fit(
        X_fit,
        y_fit,
        validation_data=(X_early, y_early),
        epochs=params["max_epochs"],
        batch_size=params["batch_size"],
        verbose=0,
        callbacks=callbacks,
        class_weight=class_weight_from_target(y_fit, params.get("class_weight_mode")),
    )
    scores = model.predict(X_eval, batch_size=params["batch_size"], verbose=0).reshape(-1)

    threshold_source = "fixed"
    if tune_threshold:
        decision_threshold, threshold_selection_balanced_accuracy = best_threshold_for_balanced_accuracy(y_eval, scores)
        threshold_source = "validation_tuned"
    else:
        threshold_selection_balanced_accuracy = np.nan
        if decision_threshold is None:
            decision_threshold = 0.5
            threshold_source = "default_0p5"

    metric_result = metrics_from_scores(y_eval, scores, float(decision_threshold))
    preds = metric_result.pop("preds")
    metadata = FEATURE_SET_METADATA.get(feature_set_name, {})
    row = {
        "split": split_name,
        "feature_set": feature_set_name,
        "feature_family": metadata.get("feature_family"),
        "volume_transform": metadata.get("volume_option"),
        "gdelt_transform": metadata.get("gdelt_option"),
        "reddit_transform": metadata.get("reddit_option"),
        "google_transform": metadata.get("google_option"),
        "features": ", ".join(features),
        "param_set": params["param_set"],
        "n_features": len(features),
        "train_rows": len(train_input_df),
        "eval_rows": len(eval_input_df),
        "fit_sequences": len(y_fit),
        "early_stop_sequences": len(y_early),
        "eval_sequences": len(y_eval),
        "eval_positive_rate": float(np.mean(y_eval)),
        "decision_threshold": float(decision_threshold),
        "threshold_source": threshold_source,
        "threshold_selection_balanced_accuracy": threshold_selection_balanced_accuracy,
        "epochs_ran": len(history.history.get("loss", [])),
        "best_val_loss": float(np.min(history.history.get("val_loss", [np.nan]))),
        **metric_result,
    }
    add_rnn_param_columns(row, params)

    if not return_predictions:
        return row

    predictions_df = eval_metadata_df.copy()
    predictions_df["score"] = scores
    predictions_df["prediction"] = preds
    predictions_df["decision_threshold"] = float(decision_threshold)
    return row, predictions_df


def evaluate_rnn_config_walk_forward(
    *,
    feature_set_name: str,
    features: list[str],
    params: dict,
    modeled_input_df: pd.DataFrame,
    fold_specs: list[dict],
) -> tuple[dict, list[dict]]:
    fold_rows = []
    for spec in fold_specs:
        fold_row = evaluate_rnn_params(
            feature_set_name=feature_set_name,
            features=features,
            params=params,
            train_input_df=subset_by_dates(modeled_input_df, spec["train_dates"]),
            eval_input_df=subset_by_dates(modeled_input_df, spec["validation_dates"]),
            split_name="walk_forward_validation",
            tune_threshold=CONFIG["tune_decision_threshold"],
        )
        fold_rows.append(
            {
                **fold_row,
                "fold": spec["fold"],
                "fold_train_n_dates": spec["train_n_dates"],
                "fold_validation_n_dates": spec["validation_n_dates"],
                "fold_train_date_min": spec["train_date_min"],
                "fold_train_date_max": spec["train_date_max"],
                "fold_validation_date_min": spec["validation_date_min"],
                "fold_validation_date_max": spec["validation_date_max"],
            }
        )

    fold_results_df = pd.DataFrame(fold_rows)
    metadata = FEATURE_SET_METADATA.get(feature_set_name, {})
    primary_metric = CONFIG["primary_validation_metric"]
    metric_mean = float(fold_results_df[primary_metric].mean())
    metric_std = float(fold_results_df[primary_metric].std(ddof=0))
    selection_score = metric_mean - CONFIG["walk_forward_stability_penalty"] * metric_std

    summary_row = {
        "split": "walk_forward_validation",
        "feature_set": feature_set_name,
        "feature_family": metadata.get("feature_family"),
        "volume_transform": metadata.get("volume_option"),
        "gdelt_transform": metadata.get("gdelt_option"),
        "reddit_transform": metadata.get("reddit_option"),
        "google_transform": metadata.get("google_option"),
        "features": ", ".join(features),
        "param_set": params["param_set"],
        "n_features": len(features),
        "n_folds": len(fold_results_df),
        "selection_score": selection_score,
        "balanced_accuracy": metric_mean,
        "balanced_accuracy_std": metric_std,
        "balanced_accuracy_min": float(fold_results_df["balanced_accuracy"].min()),
        "accuracy": float(fold_results_df["accuracy"].mean()),
        "accuracy_std": float(fold_results_df["accuracy"].std(ddof=0)),
        "f1_score": float(fold_results_df["f1_score"].mean()),
        "f1_score_std": float(fold_results_df["f1_score"].std(ddof=0)),
        "predicted_positive_rate": float(fold_results_df["predicted_positive_rate"].mean()),
        "decision_threshold": float(fold_results_df["decision_threshold"].median()),
        "epochs_ran": float(fold_results_df["epochs_ran"].mean()),
        "best_val_loss": float(fold_results_df["best_val_loss"].mean()),
        "threshold_source": "walk_forward_fold_median",
    }
    add_rnn_param_columns(summary_row, params)
    return summary_row, fold_rows


def metric_from_prediction_frame(prediction_frame: pd.DataFrame, metric_name: str) -> float:
    return float(ClassificationMetrics.compute_from_predictions(prediction_frame, metric_name))



def block_bootstrap_ci_by_date(
    prediction_frame: pd.DataFrame,
    metric_names: list[str],
    *,
    n_bootstrap: int,
    ci: float,
    random_state: int,
) -> pd.DataFrame:
    rng = np.random.default_rng(random_state)
    grouped_by_date = {date: group for date, group in prediction_frame.groupby("date")}
    dates = np.array(list(grouped_by_date.keys()), dtype=object)
    alpha = (1.0 - ci) / 2.0
    rows = []

    for metric_name in metric_names:
        samples = []
        for _ in range(n_bootstrap):
            sampled_dates = rng.choice(dates, size=len(dates), replace=True)
            sampled_frame = pd.concat([grouped_by_date[date] for date in sampled_dates], ignore_index=True)
            metric_value = metric_from_prediction_frame(sampled_frame, metric_name)
            if pd.notna(metric_value):
                samples.append(metric_value)
        rows.append(
            {
                "metric": metric_name,
                "estimate": metric_from_prediction_frame(prediction_frame, metric_name),
                "ci_lower": float(np.quantile(samples, alpha)) if samples else np.nan,
                "ci_upper": float(np.quantile(samples, 1.0 - alpha)) if samples else np.nan,
                "bootstrap_iterations": n_bootstrap,
                "block_unit": "date",
            }
        )

    return pd.DataFrame(rows)


In [ ]:
selection_metric = CONFIG["selection_metric"]
walk_forward_grid_rows = []
walk_forward_fold_rows = []

for feature_set_name in FEATURE_SETS_TO_TEST:
    features = FEATURE_SETS[feature_set_name]
    for params in RNN_PARAM_GRID:
        summary_row, fold_rows = evaluate_rnn_config_walk_forward(
            feature_set_name=feature_set_name,
            features=features,
            params=params,
            modeled_input_df=modeled_df,
            fold_specs=walk_forward_fold_specs,
        )
        walk_forward_grid_rows.append(summary_row)
        walk_forward_fold_rows.extend(fold_rows)

walk_forward_grid_results_df = pd.DataFrame(walk_forward_grid_rows)
walk_forward_fold_results_df = pd.DataFrame(walk_forward_fold_rows)
if selection_metric not in walk_forward_grid_results_df.columns:
    raise KeyError(f"Selection metric is not available: {selection_metric}")

validation_grid_results_df = walk_forward_grid_results_df.sort_values(
    [selection_metric, "balanced_accuracy", "f1_score", "accuracy", "feature_set", "param_set"],
    ascending=[False, False, False, False, True, True],
).reset_index(drop=True)

validation_grid_results_df

In [ ]:
validation_best_by_feature_set_df = (
    validation_grid_results_df.sort_values(
        ["feature_set", CONFIG["selection_metric"], "balanced_accuracy", "f1_score", "accuracy"],
        ascending=[True, False, False, False, False],
    )
    .groupby("feature_set", as_index=False)
    .head(1)
    .sort_values([CONFIG["selection_metric"], "balanced_accuracy", "f1_score", "accuracy"], ascending=False)
    .reset_index(drop=True)
)

validation_best_by_feature_set_display_df = validation_best_by_feature_set_df.rename(
    columns={
        "param_set": "best_param_set_by_walk_forward_score",
        "balanced_accuracy": "walk_forward_mean_balanced_accuracy",
        "balanced_accuracy_std": "walk_forward_std_balanced_accuracy",
        "accuracy": "walk_forward_mean_accuracy",
        "f1_score": "walk_forward_mean_f1_score",
    }
)

validation_best_by_feature_set_display_df[
    [
        "feature_set",
        "feature_family",
        "n_features",
        "best_param_set_by_walk_forward_score",
        *RNN_PARAM_COLUMNS,
        "selection_score",
        "walk_forward_mean_balanced_accuracy",
        "walk_forward_std_balanced_accuracy",
        "walk_forward_mean_accuracy",
        "walk_forward_mean_f1_score",
        "predicted_positive_rate",
    ]
]

In [ ]:
best_validation_params_df = validation_best_by_feature_set_df.copy()
final_selected_model_df = best_validation_params_df.head(1).copy()

final_selected_model_df[
    [
        "feature_set",
        "feature_family",
        "n_features",
        "param_set",
        *RNN_PARAM_COLUMNS,
        "selection_score",
        "balanced_accuracy",
        "balanced_accuracy_std",
        "balanced_accuracy_min",
        "accuracy",
        "f1_score",
        "predicted_positive_rate",
        "features",
    ]
]

In [ ]:
final_selected_walk_forward_row = final_selected_model_df.iloc[0]
final_selected_params = params_from_result_row(final_selected_walk_forward_row)
final_selected_feature_set = final_selected_walk_forward_row["feature_set"]

final_threshold_calibration_result = evaluate_rnn_params(
    feature_set_name=final_selected_feature_set,
    features=FEATURE_SETS[final_selected_feature_set],
    params=final_selected_params,
    train_input_df=train_df,
    eval_input_df=validation_df,
    split_name="validation_threshold_calibration",
    tune_threshold=True,
)
final_threshold_calibration_result_df = pd.DataFrame([final_threshold_calibration_result])

final_test_result, final_test_predictions_df = evaluate_rnn_params(
    feature_set_name=final_selected_feature_set,
    features=FEATURE_SETS[final_selected_feature_set],
    params=final_selected_params,
    train_input_df=train_df,
    eval_input_df=test_df,
    split_name="test_final_walk_forward_selected",
    decision_threshold=final_threshold_calibration_result["decision_threshold"],
    tune_threshold=False,
    return_predictions=True,
)
final_test_result.update(
    {
        "walk_forward_selection_score": final_selected_walk_forward_row["selection_score"],
        "walk_forward_mean_balanced_accuracy": final_selected_walk_forward_row["balanced_accuracy"],
        "walk_forward_std_balanced_accuracy": final_selected_walk_forward_row["balanced_accuracy_std"],
        "threshold_calibration_accuracy": final_threshold_calibration_result["accuracy"],
        "threshold_calibration_balanced_accuracy": final_threshold_calibration_result["balanced_accuracy"],
        "threshold_calibration_f1_score": final_threshold_calibration_result["f1_score"],
    }
)
final_test_result_df = pd.DataFrame([final_test_result])

final_test_metric_ci_df = block_bootstrap_ci_by_date(
    final_test_predictions_df,
    ["accuracy", "balanced_accuracy", "f1_score"],
    n_bootstrap=CONFIG["bootstrap_iterations"],
    ci=CONFIG["bootstrap_ci"],
    random_state=CONFIG["random_state"],
)

final_test_result_df[
    [
        "feature_set",
        "feature_family",
        "n_features",
        "param_set",
        *RNN_PARAM_COLUMNS,
        "walk_forward_selection_score",
        "walk_forward_mean_balanced_accuracy",
        "walk_forward_std_balanced_accuracy",
        "decision_threshold",
        "threshold_calibration_balanced_accuracy",
        "accuracy",
        "balanced_accuracy",
        "f1_score",
        "predicted_positive_rate",
    ]
]

In [ ]:
final_test_metric_ci_df

In [ ]:
threshold_calibration_rows = []
test_rows = []

for row in best_validation_params_df.to_dict(orient="records"):
    params = params_from_result_row(row)
    feature_set_name = row["feature_set"]
    calibration_row = evaluate_rnn_params(
        feature_set_name=feature_set_name,
        features=FEATURE_SETS[feature_set_name],
        params=params,
        train_input_df=train_df,
        eval_input_df=validation_df,
        split_name="validation_threshold_calibration",
        tune_threshold=True,
    )
    threshold_calibration_rows.append(calibration_row)
    test_rows.append(
        evaluate_rnn_params(
            feature_set_name=feature_set_name,
            features=FEATURE_SETS[feature_set_name],
            params=params,
            train_input_df=train_df,
            eval_input_df=test_df,
            split_name="test_walk_forward_selected_threshold",
            decision_threshold=calibration_row["decision_threshold"],
            tune_threshold=False,
        )
    )

threshold_calibration_results_df = pd.DataFrame(threshold_calibration_rows)
test_best_validation_params_df = pd.DataFrame(test_rows).sort_values(
    ["balanced_accuracy", "f1_score", "accuracy", "feature_set"],
    ascending=[False, False, False, True],
).reset_index(drop=True)

(
    simple_hyperparameter_summary_df,
    baseline_walk_forward_row,
    baseline_calibration_row,
    baseline_test_row,
) = ModelReportBuilder.build_simple_hyperparameter_summary(
    best_validation_params_df=best_validation_params_df,
    threshold_calibration_results_df=threshold_calibration_results_df,
    test_best_validation_params_df=test_best_validation_params_df,
    baseline_feature_set=BASELINE_FEATURE_SET,
)

summary_columns = ModelReportBuilder.simple_summary_columns(
    param_columns=RNN_PARAM_COLUMNS,
    extra_columns=[],
)

# Appendix table: all feature-set winners are walk-forward selected; test sorting below is descriptive only.
simple_hyperparameter_summary_df[
    ModelReportBuilder.available_columns(simple_hyperparameter_summary_df, summary_columns)
]


In [ ]:
alternative_data_research_question_df = ModelReportBuilder.build_alternative_data_research_question(
    simple_hyperparameter_summary_df
)

alternative_data_research_question_df


In [ ]:
attention_research_question_df = ModelReportBuilder.build_attention_research_question(
    simple_hyperparameter_summary_df,
    param_columns=RNN_PARAM_COLUMNS,
)

attention_research_question_df


In [ ]:
# Descriptive comparison: each row is walk-forward selected and uses its fixed holdout-validation threshold on test.
best_validation_selected_model_by_family_test_accuracy_df = ModelReportBuilder.build_best_model_by_family(
    simple_hyperparameter_summary_df,
    sort_by_accuracy_first=True,
)

family_comparison_columns = ModelReportBuilder.family_comparison_columns(
    param_columns=RNN_PARAM_COLUMNS,
    extra_columns=[],
    compact=True,
)

best_validation_selected_model_by_family_test_accuracy_df[
    ModelReportBuilder.available_columns(best_validation_selected_model_by_family_test_accuracy_df, family_comparison_columns)
]


In [ ]:
selected_walk_forward_training_diagnostics_df = walk_forward_fold_results_df[
    walk_forward_fold_results_df["feature_set"].eq(final_selected_feature_set)
    & walk_forward_fold_results_df["param_set"].eq(final_selected_params["param_set"])
][
    [
        "fold",
        "feature_set",
        "param_set",
        "sequence_length",
        "cell_type",
        "hidden_units",
        "fit_sequences",
        "early_stop_sequences",
        "eval_sequences",
        "epochs_ran",
        "best_val_loss",
        "balanced_accuracy",
        "f1_score",
    ]
].sort_values("fold").reset_index(drop=True)

selected_walk_forward_training_diagnostics_df
